# 05 Historical fact


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
R=f"{PROJECT_ID}.nbu_raw.raw_rates"; D=f"{PROJECT_ID}.nbu_dwh.dim_currency"; T=f"{PROJECT_ID}.nbu_dwh.fact_exchange_rate"; r=client.query(f"SELECT ingested_at,business_date,payload FROM `{R}` WHERE business_date=CURRENT_DATE()").to_dataframe(); assert not r.empty; p=pd.json_normalize(r.payload.map(json.loads)); d=pd.concat([r[["ingested_at","business_date"]].reset_index(drop=True),p],axis=1); d["currency_code"]=d.cc.astype(str).str.strip().str.upper(); d["rate"]=pd.to_numeric(d.rate); n=d.sort_values("ingested_at").drop_duplicates(["business_date","currency_code"],keep="last")[["business_date","currency_code","rate"]].merge(client.query(f"SELECT currency_key,currency_code FROM `{D}`").to_dataframe(),on="currency_code",how="left"); n.currency_key=n.currency_key.fillna(-1).astype(int); n["date_key"]=pd.to_datetime(n.business_date).dt.strftime("%Y%m%d").astype(int); n["dw_load_ts"]=pd.Timestamp.now(tz="UTC"); n=n[["date_key","currency_key","rate","business_date","dw_load_ts"]]
s=[bigquery.SchemaField("date_key","INTEGER"),bigquery.SchemaField("currency_key","INTEGER"),bigquery.SchemaField("rate","FLOAT"),bigquery.SchemaField("business_date","DATE"),bigquery.SchemaField("dw_load_ts","TIMESTAMP")]; t=bigquery.Table(T,schema=s); t.time_partitioning=bigquery.TimePartitioning(field="business_date"); t.clustering_fields=["currency_key"]; client.create_table(t,exists_ok=True); old=client.query(f"SELECT * FROM `{T}` WHERE business_date<>CURRENT_DATE()").to_dataframe(); m=pd.concat([old,n],ignore_index=True).drop_duplicates(["date_key","currency_key"],keep="last"); client.load_table_from_dataframe(m,T,job_config=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_TRUNCATE")).result(); dup=client.query(f"SELECT date_key,currency_key,COUNT(*) c FROM `{T}` GROUP BY 1,2 HAVING c>1").to_dataframe(); assert dup.empty; display(client.query(f"SELECT COUNT(DISTINCT business_date) business_dates,COUNT(*) total_rows FROM `{T}`").to_dataframe())